# Expresiones Regulares (Regex) en Ciencia de Datos

## 🎯 Objetivos de Aprendizaje
- Comprender el motor de búsqueda por patrones de expresiones regulares en Python (`re`).
- Dominar metacaracteres, cuantificadores, clases de caracteres y anclas de posición.
- Aplicar grupos de captura y named groups (`(?P<nombre>...)`) para desestructurar texto no estructurado.
- Integrar expresiones regulares de forma vectorizada en Pandas con `.str.extract()`, `.str.contains()` y `.str.replace()`.
- Validar y limpiar campos críticos como emails, teléfonos, fechas y códigos postales.

## 🌉 Puente Pedagógico: El Microscopio de Patrones de Texto

### ¿Por qué importa Regex?
Gran parte de los datos corporativos provienen de logs, descripciones libres o scraping web donde los números y categorías están mezclados con texto libre. Regex permite formular reglas precisas para aislar exactamente los fragmentos de interés.

### Analogía
Imagina un colador de laboratorio:
- Una búsqueda tradicional (`"perro" in texto`) solo busca una palabra exacta fija.
- Una expresión regular es una malla con ranuras específicas que deja pasar cualquier palabra que empiece con mayúscula, seguida de 3 números y termine con un guión.

### Diagrama ASCII: Anatomía de un Patrón Regex
```
   Patrón:  ^([A-Z]{3})-(\d{4})\b
            | |      | |  |    |
            | |      | |  |    +--> Límite de palabra (Word boundary)
            | |      | |  +-------> Exactamente 4 dígitos (0-9)
            | |      | +----------> Carácter literal guión medio (-)
            | |      +------------> Exactamente 3 letras mayúsculas
            | +-------------------> Grupo de captura 1
            +---------------------> Inicio de línea / string
```

In [ ]:
import re
import pandas as pd

texto_ejemplo = """
Contacto de soporte: soporte@empresa.com, fecha: 2026-04-10, ticket #8942.
Ventas: ventas.latam@negocio.org, fecha: 2026-05-12, ticket #1209.
Urgencias: admin_12@datacenter.net, fecha: 2026-06-01, ticket #0045.
"""

# Extracción de correos electrónicos con regex
patron_email = r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"
emails = re.findall(patron_email, texto_ejemplo)
print("Emails encontrados:", emails)

## 1. Tabla Maestra de Metacaracteres

| Símbolo | Significado | Ejemplo | Coincide con |
|:---|:---|:---|:---|
| `.` | Cualquier carácter salvo salto de línea | `a.c` | `abc`, `a1c` |
| `\d` / `\D` | Dígito `[0-9]` / No dígito | `\d{3}` | `123`, `999` |
| `\w` / `\W` | Alfanumérico `[a-zA-Z0-9_]` / No alfanumérico | `\w+` | `palabra_1` |
| `\s` / `\S` | Espacio en blanco (`\t`, `\n`, espacio) / No espacio | `\s+` | Espacios |
| `^` / `$` | Inicio de línea / Fin de línea | `^Inicio` | Texto al principio |
| `+` / `*` / `?` | 1 o más / 0 o más / Opcional (0 o 1) | `colou?r` | `color`, `colour` |
| `{n,m}` | Entre $n$ y $m$ repeticiones | `\d{2,4}` | `12`, `1234` |
| `(...)` | Grupo de captura | `(\d{4})-(\d{2})` | Grupos separados |

## 2. Integración Vectorizada con Pandas: `.str.extract()`

In [ ]:
df_logs = pd.DataFrame({
    "log_raw": [
        "ID: 4501 - ERROR: Disco lleno (Cod: ERR_99)",
        "ID: 4502 - INFO: Usuario logueado (Cod: INF_01)",
        "ID: 4503 - WARN: Memoria sobre 85% (Cod: WRN_42)"
    ]
})

# Extracción de campos con grupos nombrados (?P<nombre>...)
patron_log = r"ID:\s*(?P<id>\d+)\s*-\s*(?P<nivel>[A-Z]+):\s*(?P<mensaje>.*)\s*\(Cod:\s*(?P<codigo>[A-Z_0-9]+)\)"
df_parsed = df_logs["log_raw"].str.extract(patron_log)
display(df_parsed)

## 📝 Ejercicios Prácticos

### Ejercicio 1 (Guiado): Enmascaramiento de datos personales
Reemplaza las tarjetas de crédito simuladas por asteriscos dejando solo los últimos 4 dígitos.

In [ ]:
tarjetas = ["4532-8901-2345-6789", "5412-0098-7654-3210"]
enmascaradas = [re.sub(r"\d{4}-\d{4}-\d{4}-(\d{4})", r"****-****-****-\1", t) for t in tarjetas]
print("Tarjetas protegidas:", enmascaradas)

### Ejercicio 2 (Independiente): Extracción de Dominios Web
Extrae el dominio (`empresa.com`) de la lista de correos usando `.str.extract()`.

In [ ]:
df_contactos = pd.DataFrame({"email": ["carlos@gmail.com", "laura@corporativo.org", "luis@tech.ai"]})
# Solución:
df_contactos["dominio"] = df_contactos["email"].str.extract(r"@([a-zA-Z0-9.-]+)")
display(df_contactos)

## 📋 Resumen
- Las cadenas crudas (`r"..."`) previenen el escape accidental de barras invertidas en Python.
- Los grupos de captura nombrados permiten convertir strings caóticos en DataFrames estructurados con nombres de columna limpios en un solo paso.
- Usar expresiones regulares precompiladas (`re.compile()`) cuando se evalúan millones de strings acelera sensiblemente el rendimiento.